In [ ]:
#Agent avec Web Search Tool
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage,SystemMessage
from langchain.agents import create_agent



load_dotenv()


# Initialiser le modèle
model = ChatOpenAI(
    model="gpt-5.2",  # gpt-5.2 n'existe pas
    temperature=0.7
)

tavily_client = TavilyClient()

@tool()
def web_search(query: str) -> Dict[str, Any]:
    """Recherche les recettes de cuisine correspondant aux ingrédient."""
    return tavily_client.search(query)
system_prompt = """
Vous êtes un chef de cuisine personnel qui reçoit une liste d'ingrédient et qui propose 
une liste de plats adapté aux ingrédients disponible
Veuillez respecter la structure ci-dessous.
Ingrédients : liste des ingédients
Plats : liste des plats
"""

agent = create_agent(model=model,tools=[web_search],system_prompt=system_prompt,checkpointer=InMemorySaver(),)

question = HumanMessage(content="""Liste des Ingrédients:farine,
,bsucreeurre,lait,oeuf(s),eau,sel .""")

config = {"configurable": {"thread_id": "1"}}
response = agent.invoke({"messages": [question]},config,)

question = HumanMessage(content="Quel est mon métier ?")
response = agent.invoke({"messages": [question]},config,)
print(response['messages'][-1].content)





Ingrédients : farine, sucre, beurre, lait, œuf(s), eau, sel

Plats :
- Crêpes (farine, lait, œufs, sucre, sel, beurre)
- Pancakes (farine, lait, œufs, sucre, beurre, sel)
- Gaufres simples (farine, lait, œufs, sucre, beurre, sel)
- Quatre-quarts (farine, sucre, beurre, œufs, sel)
- Cake nature (farine, beurre, sucre, œufs, lait, sel)
- Biscuits sablés (farine, beurre, sucre, œufs, sel)
- Pâte brisée (farine, beurre, eau, sel) – pour tartes/quiches
- Pâte à tarte sucrée (farine, beurre, sucre, œufs, sel)
- Beignets / pâte à frire simple (farine, eau ou lait, œufs, sel, sucre)
- Galettes/flatbread à la poêle (farine, eau, sel, un peu de beurre)


In [8]:
question = HumanMessage(content="Quel est la liste des plats avec les ingrédients précédents ?")
response = agent.invoke({"messages": [question]},config,)
print(response['messages'][-1].content)


Ingrédients : farine, sucre, beurre, lait, œuf(s), eau, sel

Plats :
- Crêpes
- Pancakes
- Gaufres
- Quatre-quarts
- Cake nature
- Biscuits sablés
- Cookies nature
- Beignets (pâte à beignets simple)
- Galettes à la poêle (pain plat sans levure)
- Pâte à choux nature (choux/éclairs nature)
- Crème pâtissière
- Pâte brisée (fond de tarte)
- Pâte sucrée (fond de tarte sucrée)
